In [69]:

import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# ============================================================
# PARAMETERS
# ============================================================
TICKERS = ["TQQQ", "CRWV", "AAL", "AMZN", "OCGN", "NIO", "ORCL"]  # Add/remove tickers here
WINDOW_DAYS = 5  # Change to 1, 5, 30, etc.
TOP_N = 1         # Number of worst/best windows to display
LOOKBACK_DAYS = 180
SHOW = "both"     # "worst", "best", or "both"
# ============================================================

end_date = datetime.today()
start_date = end_date - timedelta(days=LOOKBACK_DAYS)

label = "1-Day" if WINDOW_DAYS == 1 else f"{WINDOW_DAYS}-Day"
col_name = f"{WINDOW_DAYS}Day_Return_%"

# Store target prices for cell 2
TARGET_PRICES = {}

def build_window_table(idx_list, df, col_name, window_days, label, current_price=None):
    rows = []
    for end_date_idx in idx_list:
        ep = df.index.get_loc(end_date_idx)
        sp = max(ep - window_days, 0)
        start_date_idx = df.index[sp]
        ret = round(float(df.loc[end_date_idx, col_name]), 2)
        row = {
            "End Date": end_date_idx.date(),
            "Open Price ($)": round(float(df.loc[start_date_idx, "Open"]), 2),
            "Close Price ($)": round(float(df.loc[end_date_idx, "Close"]), 2),
            f"{label} Return (%)": ret
        }
        if window_days > 1:
            row = {"Start Date": start_date_idx.date(), **row}
        if current_price is not None:
            row["Current ($)"] = round(current_price, 2)
            row["Target ($)"] = round(current_price * (1 + ret / 100), 2)
        rows.append(row)
    return pd.DataFrame(rows)

all_rows = []
for ticker in TICKERS:
    df = yf.download(ticker, start=start_date.strftime("%Y-%m-%d"), end=end_date.strftime("%Y-%m-%d"), progress=False)
    if df.empty:
        print(f"No data found for {ticker}, skipping.")
        continue

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    current_price = float(df["Close"].iloc[-1])

    df[col_name] = df["Close"].pct_change(periods=WINDOW_DAYS) * 100

    # CALL row (best window)
    if SHOW in ("best", "both"):
        best_idx = df[col_name].idxmax()
        best_ret = round(float(df.loc[best_idx, col_name]), 2)
        best_ep = df.index.get_loc(best_idx)
        best_sp = max(best_ep - WINDOW_DAYS, 0)
        all_rows.append({
            "Ticker": ticker,
            "Analysis": f"{label} Rolling",
            "Type": "CALL",
            f"{label} Return (%)": best_ret,
            "Current ($)": round(current_price, 2),
            "Target ($)": round(current_price * (1 + best_ret / 100), 2),
            "Window": f"{df.index[best_sp].date()} \u2192 {best_idx.date()}"
        })
        TARGET_PRICES.setdefault(ticker, {})["call"] = round(current_price * (1 + best_ret / 100), 2)

    # PUT row (worst window)
    if SHOW in ("worst", "both"):
        worst_idx = df[col_name].idxmin()
        worst_ret = round(float(df.loc[worst_idx, col_name]), 2)
        worst_ep = df.index.get_loc(worst_idx)
        worst_sp = max(worst_ep - WINDOW_DAYS, 0)
        all_rows.append({
            "Ticker": ticker,
            "Analysis": f"{label} Rolling",
            "Type": "PUT",
            f"{label} Return (%)": worst_ret,
            "Current ($)": round(current_price, 2),
            "Target ($)": round(current_price * (1 + worst_ret / 100), 2),
            "Window": f"{df.index[worst_sp].date()} \u2192 {worst_idx.date()}"
        })
        TARGET_PRICES.setdefault(ticker, {})["put"] = round(current_price * (1 + worst_ret / 100), 2)

if all_rows:
    print(pd.DataFrame(all_rows).to_string(index=False))


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


  TQQQ — 5-Day Rolling Window Analysis
Type  5-Day Return (%)  Current ($)  Target ($)                  Window
CALL             17.42        46.83       54.99 2025-11-20 → 2025-11-28
 PUT            -14.99        46.83       39.81 2026-01-29 → 2026-02-05

  CRWV — 5-Day Rolling Window Analysis
Type  5-Day Return (%)  Current ($)  Target ($)                  Window
CALL             28.20        79.86      102.38 2026-02-05 → 2026-02-12
 PUT            -28.67        79.86       56.96 2025-11-10 → 2025-11-17

  AAL — 5-Day Rolling Window Analysis
Type  5-Day Return (%)  Current ($)  Target ($)                  Window
CALL             16.19        10.55       12.26 2025-10-17 → 2025-10-24
 PUT            -15.42        10.55        8.92 2026-02-26 → 2026-03-05

  AMZN — 5-Day Rolling Window Analysis
Type  5-Day Return (%)  Current ($)  Target ($)                  Window
CALL             11.91       209.53      234.49 2025-10-27 → 2025-11-03
 PUT            -14.09       209.53      180.01 2

In [76]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# ============================================================
# PARAMETERS
# ============================================================
# Uses TICKERS from cell 1, or override here
# TICKERS = ["RIVN", "TQQQ", "AMZN"]
today = datetime.today()
days_until_friday = (4 - today.weekday()) % 7 or 7  # 4 = Friday
EXPIRATION = "2026-03-20"
# EXPIRATION = (today + timedelta(days=days_until_friday)).strftime("%Y-%m-%d")
PREMIUM_PCT = 0.005  # 0.5% of current price
ONLY_ORDERS = False   # True = show only ORDER rows, False = show all
# ============================================================

def find_closest_option(df, target):
    df = df.copy()
    df = df[df["bid"] > 0]
    if df.empty:
        return None
    df["mid_price"] = (df["bid"] + df["ask"]) / 2
    df["diff"] = (df["mid_price"] - target).abs()
    closest = df.loc[df["diff"].idxmin()]
    return closest

for ticker in TICKERS:
    stock = yf.Ticker(ticker)
    current_price = stock.fast_info["lastPrice"]
    target_premium = round(current_price * PREMIUM_PCT, 2)
    header = f"\n{ticker}  |  Price: ${current_price:.2f}  |  Exp: {EXPIRATION}  |  Target premium: ${target_premium:.2f} ({PREMIUM_PCT*100:.1f}%)"

    try:
        opt_chain = stock.option_chain(EXPIRATION)
    except Exception as e:
        print(f"  Could not load options for {ticker}: {e}")
        continue

    closest_call = find_closest_option(opt_chain.calls, target_premium)
    closest_put = find_closest_option(opt_chain.puts, target_premium)

    rows = []
    targets = TARGET_PRICES.get(ticker, {})
    for lbl, opt, tgt_key in [("CALL", closest_call, "call"), ("PUT", closest_put, "put")]:
        if opt is None:
            continue
        strike = round(float(opt["strike"]), 2)
        target = targets.get(tgt_key, None)
        signal = ""
        if target is not None:
            if lbl == "CALL" and strike > target:
                signal = "ORDER"
            elif lbl == "PUT" and strike < target:
                signal = "ORDER"
        row = {
            "Type": lbl,
            "Strike ($)": strike,
            "Bid ($)": round(float(opt["bid"]), 2),
            "Ask ($)": round(float(opt["ask"]), 2),
            "Mid ($)": round(float(opt["mid_price"]), 2),
            "Target ($)": target if target is not None else "N/A",
            "Signal": signal
        }
        rows.append(row)
    result_df = pd.DataFrame(rows)
    if ONLY_ORDERS:
        result_df = result_df[result_df["Signal"] == "ORDER"]
    if not result_df.empty:
        print(header)
        print(result_df.to_string(index=False))



TQQQ  |  Price: $45.93  |  Exp: 2026-03-20  |  Target premium: $0.23 (0.5%)
Type  Strike ($)  Bid ($)  Ask ($)  Mid ($)  Target ($) Signal
CALL        51.0     0.25     0.26     0.26       54.99       
 PUT        37.0     0.22     0.24     0.23       39.81  ORDER

CRWV  |  Price: $81.11  |  Exp: 2026-03-20  |  Target premium: $0.41 (0.5%)
Type  Strike ($)  Bid ($)  Ask ($)  Mid ($)  Target ($) Signal
CALL        98.0     0.33     0.46     0.40      102.38       
 PUT        65.0     0.35     0.40     0.38       56.96       

AAL  |  Price: $10.30  |  Exp: 2026-03-20  |  Target premium: $0.05 (0.5%)
Type  Strike ($)  Bid ($)  Ask ($)  Mid ($)  Target ($) Signal
CALL        12.0     0.04     0.05     0.04       12.26       
 PUT         9.0     0.08     0.09     0.08        8.92       

AMZN  |  Price: $207.67  |  Exp: 2026-03-20  |  Target premium: $1.04 (0.5%)
Type  Strike ($)  Bid ($)  Ask ($)  Mid ($)  Target ($) Signal
CALL       215.0     1.23     1.30     1.27      234.49       